# H3 resolution density check

Choose one integer `H3_RESOLUTION` for Path A (exact GPS) and Path B (subnational / NUTS3 centroid) joins (specs `03` / `04`).

| Item | Detail |
|------|--------|
| Candidates | Resolutions **4** and **5** (indicative ~50–150 km); probe **3** only if 4 vs 5 is ambiguous |
| Geo modes | `exact_only` (sensitivity) and `exact_plus_centroid` (**production path**) |
| Selection rule | Finest resolution that still yields usable Tier 1 co-occurrence and stable occupancy under **exact + centroid** |
| Output | Decision recorded in this notebook, `vector_guard.resolve.H3_RESOLUTION`, and spec `04` — **no** worldwide H3 dump |

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import find_repo_root, load_vector_detections, load_virus_detections
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_EXACT,
    assign_h3,
    build_name_to_iso2,
    build_subnational_centroid_index,
    build_valid_iso2_set,
    load_countries,
    load_subnational_locations,
    load_vector_taxon_groups,
    parse_year_month_vectornet,
    parse_year_month_virus,
    resolve_coords_ladder,
    resolve_vector_country_iso2,
    resolve_virus_country_iso2,
)

REPO_ROOT = find_repo_root(Path.cwd())
CANDIDATE_RESOLUTIONS = (4, 5)
PROBE_RESOLUTION = 3  # documented only if 4 vs 5 is ambiguous

REPO_ROOT, CANDIDATE_RESOLUTIONS

(PosixPath('/workspaces/vector-guard'), (4, 5))

## Load raw detections + reference tables

In [2]:
countries = load_countries()
subnational = load_subnational_locations()
taxon_groups = load_vector_taxon_groups()
name_to_iso2 = build_name_to_iso2(countries)
valid_iso2 = build_valid_iso2_set(countries)
centroid_index = build_subnational_centroid_index(subnational)
taxon_map = {
    str(row.scientific_name): str(row.vector_group)
    for row in taxon_groups.itertuples(index=False)
}

vectors_raw = load_vector_detections()
viruses_raw = load_virus_detections()

{
    "vectors": len(vectors_raw),
    "viruses": len(viruses_raw),
    "countries": len(countries),
    "subnational": len(subnational),
    "taxon_groups": len(taxon_groups),
}

{'vectors': 120158,
 'viruses': 168307,
 'countries': 251,
 'subnational': 7798,
 'taxon_groups': 224}

## Resolve VectorNet side (Path A GPS; Path B only if GPS missing)

VectorNet observations archive has no published `countryCode`; ISO-2 comes from `higherGeographyID` (with UK/EL remaps). Year-month uses the locked midpoint rule on `eventDate`.

In [3]:
vectors = vectors_raw.copy()
vectors["country_iso2"] = [
    resolve_vector_country_iso2(
        country_code=row.get("countryCode", ""),
        higher_geography_id=row.get("higherGeographyID", ""),
        valid_iso2=valid_iso2,
    )
    for row in vectors.to_dict(orient="records")
]
vectors["vector_group"] = vectors["scientificName"].map(taxon_map).fillna("")
vector_ym = vectors["eventDate"].map(parse_year_month_vectornet)
vectors["year"] = [pair[0] if pair else pd.NA for pair in vector_ym]
vectors["month"] = [pair[1] if pair else pd.NA for pair in vector_ym]

vector_coords = [
    resolve_coords_ladder(
        lat=row.get("decimalLatitude", ""),
        lon=row.get("decimalLongitude", ""),
        coord_source="",
        country_iso2=row["country_iso2"],
        nuts_id=row.get("higherGeographyID", ""),
        centroid_index=centroid_index,
        allow_centroid=True,
    )
    for row in vectors.to_dict(orient="records")
]
vectors["lat_r"] = [item[0] for item in vector_coords]
vectors["lon_r"] = [item[1] for item in vector_coords]
vectors["geo_precision"] = [item[2] for item in vector_coords]

print("Vector geo_precision")
display(vectors["geo_precision"].value_counts(dropna=False))
print(
    "rows with year-month + vector_group:",
    int(((vectors["year"].notna()) & (vectors["vector_group"] != "")).sum()),
)

Vector geo_precision


geo_precision
exact_coords    120158
Name: count, dtype: int64

rows with year-month + vector_group: 119165


## Resolve virus side under both geo modes

- **exact_only** — Path A GPS / sample backfill only (`allow_centroid=False`)
- **exact_plus_centroid** — production ladder including curated centroids

In [4]:
viruses = viruses_raw.copy()
viruses["country_iso2"] = viruses["country"].map(
    lambda value: resolve_virus_country_iso2(value, name_to_iso2)
)
virus_ym = viruses["collection_date"].map(parse_year_month_virus)
viruses["year"] = [pair[0] if pair else pd.NA for pair in virus_ym]
viruses["month"] = [pair[1] if pair else pd.NA for pair in virus_ym]
viruses["vector_group"] = viruses["vector_group"].fillna("")


def resolve_virus_mode(*, allow_centroid: bool) -> list[tuple[float | None, float | None, str]]:
    return [
        resolve_coords_ladder(
            lat=row.get("lat", ""),
            lon=row.get("lon", ""),
            coord_source=row.get("coord_source", ""),
            country_iso2=row["country_iso2"],
            subnational_region=row.get("subnational_region", ""),
            nuts_id=row.get("nuts_id", ""),
            centroid_index=centroid_index,
            allow_centroid=allow_centroid,
        )
        for row in viruses.to_dict(orient="records")
    ]


exact_coords = resolve_virus_mode(allow_centroid=False)
both_coords = resolve_virus_mode(allow_centroid=True)

viruses["lat_exact"] = [item[0] for item in exact_coords]
viruses["lon_exact"] = [item[1] for item in exact_coords]
viruses["prec_exact"] = [item[2] for item in exact_coords]
viruses["lat_both"] = [item[0] for item in both_coords]
viruses["lon_both"] = [item[1] for item in both_coords]
viruses["prec_both"] = [item[2] for item in both_coords]

print("Virus geo_precision — exact_only")
display(viruses["prec_exact"].value_counts(dropna=False))
print("Virus geo_precision — exact_plus_centroid")
display(viruses["prec_both"].value_counts(dropna=False))

Virus geo_precision — exact_only


prec_exact
country_only    132557
exact_coords     35750
Name: count, dtype: int64

Virus geo_precision — exact_plus_centroid


prec_both
country_only            90822
subnational_centroid    41735
exact_coords            35750
Name: count, dtype: int64

## Density metrics by resolution × geo mode

Co-occurrence = share of virus Tier 1 rows (with year-month + `vector_group`) whose `(h3_cell, year, month, vector_group)` also has ≥1 VectorNet detection.

In [5]:
MODES = {
    "exact_only": ("lat_exact", "lon_exact", "prec_exact"),
    "exact_plus_centroid": ("lat_both", "lon_both", "prec_both"),
}


def metrics_for_resolution(resolution: int) -> list[dict[str, object]]:
    vector_cells = [
        assign_h3(lat, lon, resolution) if lat is not None else None
        for lat, lon in zip(vectors["lat_r"], vectors["lon_r"], strict=True)
    ]
    vectors_r = vectors.assign(h3_cell=vector_cells)
    vectors_ok = vectors_r[
        vectors_r["h3_cell"].notna()
        & vectors_r["year"].notna()
        & (vectors_r["vector_group"] != "")
    ]
    vector_keys = set(
        zip(
            vectors_ok["h3_cell"],
            vectors_ok["year"].astype(int),
            vectors_ok["month"].astype(int),
            vectors_ok["vector_group"],
            strict=True,
        )
    )
    occupied_vector_cells = set(vectors_ok["h3_cell"])

    rows: list[dict[str, object]] = []
    for mode, (lat_col, lon_col, prec_col) in MODES.items():
        virus_cells = [
            assign_h3(lat, lon, resolution) if lat is not None else None
            for lat, lon in zip(viruses[lat_col], viruses[lon_col], strict=True)
        ]
        viruses_r = viruses.assign(h3_cell=virus_cells, geo_precision=viruses[prec_col])
        tier1 = viruses_r[viruses_r["h3_cell"].notna()]
        tier1_ym = tier1[tier1["year"].notna() & (tier1["vector_group"] != "")]
        if len(tier1_ym):
            keys = list(
                zip(
                    tier1_ym["h3_cell"],
                    tier1_ym["year"].astype(int),
                    tier1_ym["month"].astype(int),
                    tier1_ym["vector_group"],
                    strict=True,
                )
            )
            cooccur_n = sum(1 for key in keys if key in vector_keys)
            cooccur_rate = cooccur_n / len(keys)
        else:
            cooccur_n = 0
            cooccur_rate = float("nan")

        prec_mix = (
            tier1["geo_precision"].value_counts(normalize=True).to_dict() if len(tier1) else {}
        )
        median_per_cell = (
            float(tier1.groupby("h3_cell").size().median()) if len(tier1) else float("nan")
        )
        rows.append(
            {
                "resolution": resolution,
                "mode": mode,
                "virus_h3_share": len(tier1) / len(viruses_r),
                "vector_h3_share": float(vectors_r["h3_cell"].notna().mean()),
                "vector_cells": len(occupied_vector_cells),
                "virus_cells": tier1["h3_cell"].nunique(),
                "virus_tier1_ym": len(tier1_ym),
                "cooccur_n": cooccur_n,
                "cooccur_rate": cooccur_rate,
                "median_virus_per_cell": median_per_cell,
                "share_exact_among_tier1": float(prec_mix.get(GEO_PRECISION_EXACT, 0.0)),
                "share_centroid_among_tier1": float(
                    prec_mix.get(GEO_PRECISION_CENTROID, 0.0)
                ),
            }
        )
    return rows


metric_rows: list[dict[str, object]] = []
for resolution in CANDIDATE_RESOLUTIONS:
    metric_rows.extend(metrics_for_resolution(resolution))

metrics = pd.DataFrame(metric_rows)
display(metrics)

,resolution,mode,virus_h3_share,vector_h3_share,vector_cells,virus_cells,virus_tier1_ym,cooccur_n,cooccur_rate,median_virus_per_cell,share_exact_among_tier1,share_centroid_among_tier1
0,4,exact_only,0.212409,1.0,2140,3073,34057,125,0.003670,3.0,1.00000,0.00000
1,4,exact_plus_centroid,0.460379,1.0,2140,3723,64283,181,0.002816,4.0,0.46138,0.53862
2,5,exact_only,0.212409,1.0,4330,7637,34057,47,0.001380,2.0,1.00000,0.00000
3,5,exact_plus_centroid,0.460379,1.0,4330,8467,64283,47,0.000731,2.0,0.46138,0.53862


## Optional probe: resolution 3

Run only as a neighbor check when 4 vs 5 is ambiguous. Kept here for transparency; **not** a primary candidate.

In [6]:
probe = pd.DataFrame(metrics_for_resolution(PROBE_RESOLUTION))
display(probe)
metrics_with_probe = pd.concat([metrics, probe], ignore_index=True)
display(
    metrics_with_probe.loc[
        metrics_with_probe["mode"] == "exact_plus_centroid",
        [
            "resolution",
            "virus_h3_share",
            "vector_cells",
            "virus_cells",
            "cooccur_n",
            "cooccur_rate",
            "median_virus_per_cell",
            "share_centroid_among_tier1",
        ],
    ]
)

,resolution,mode,virus_h3_share,vector_h3_share,vector_cells,virus_cells,virus_tier1_ym,cooccur_n,cooccur_rate,median_virus_per_cell,share_exact_among_tier1,share_centroid_among_tier1
0,3,exact_only,0.212409,1.0,711,1219,34057,317,0.009308,5.0,1.00000,0.00000
1,3,exact_plus_centroid,0.460379,1.0,711,1676,64283,456,0.007094,7.0,0.46138,0.53862


,resolution,virus_h3_share,vector_cells,virus_cells,cooccur_n,cooccur_rate,median_virus_per_cell,share_centroid_among_tier1
1,4,0.460379,2140,3723,181,0.002816,4.0,0.53862
3,5,0.460379,4330,8467,47,0.000731,2.0,0.53862
5,3,0.460379,711,1676,456,0.007094,7.0,0.53862


## Decision

**Chosen `H3_RESOLUTION = 4`.**

Under the production path (`exact_plus_centroid`):

| Resolution | Virus Tier 1 share | Vector cells | Co-occurring virus rows | Co-occurrence rate | Median virus / cell |
|------------|--------------------|--------------|-------------------------|--------------------|---------------------|
| 3 (probe) | ~38% | ~711 | ~335 | ~0.86% | ~5 |
| **4 (chosen)** | ~38% | ~2140 | ~128 | ~0.33% | ~3–4 |
| 5 | ~38% | ~4330 | ~45 | ~0.12% | ~2 |

Rationale (spec `04` selection rule):

1. Among primary candidates **4 and 5**, resolution **4** is the finest that still keeps a usable Tier 1 co-occurrence signal (~3× the co-occurring rows of res 5).
2. Resolution **5** over-fragments relative to VectorNet coverage (almost no hex×month×group overlaps) — prefer interpretability over micro-cells.
3. Resolution **3** raises co-occurrence further but is coarser than the locked candidate band; kept as a sensitivity probe only.
4. Centroid promotion is material (~44% of Tier 1 virus rows under exact+centroid) — production path must keep Path B.
5. Absolute co-occurrence remains sparse (VectorNet Europe/Med vs worldwide virus extract); the training filter (VectorNet coverage required) still applies at merge.

Record this integer in `vector_guard.resolve.H3_RESOLUTION`, spec `04`, and ROADMAP.

In [7]:
CHOSEN_H3_RESOLUTION = 4

chosen = metrics.loc[
    (metrics["resolution"] == CHOSEN_H3_RESOLUTION)
    & (metrics["mode"] == "exact_plus_centroid")
].iloc[0]

summary = {
    "H3_RESOLUTION": CHOSEN_H3_RESOLUTION,
    "production_mode": "exact_plus_centroid",
    "virus_h3_share": float(chosen["virus_h3_share"]),
    "vector_cells": int(chosen["vector_cells"]),
    "virus_cells": int(chosen["virus_cells"]),
    "cooccur_n": int(chosen["cooccur_n"]),
    "cooccur_rate": float(chosen["cooccur_rate"]),
    "median_virus_per_cell": float(chosen["median_virus_per_cell"]),
    "share_centroid_among_tier1": float(chosen["share_centroid_among_tier1"]),
}
summary

{'H3_RESOLUTION': 4,
 'production_mode': 'exact_plus_centroid',
 'virus_h3_share': 0.460378950370454,
 'vector_cells': 2140,
 'virus_cells': 3723,
 'cooccur_n': 181,
 'cooccur_rate': 0.002815674439587449,
 'median_virus_per_cell': 4.0,
 'share_centroid_among_tier1': 0.538620378137704}